# Contrôle qualité des données

Résultats des contrôles qualité et réconciliation sur le Data Warehouse du projet (il faut avoir lancé l'ETL). La preuve que les règles de rejet fonctionnent est dans `test_rejets.ipynb`.

In [1]:
from pathlib import Path

# Un notebook n'a pas de __file__ : on remonte les dossiers jusqu'à la racine du projet
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "03_data" / "raw").is_dir())
print("Projet :", ROOT.name)

Projet : SmartPort_Projet01


## Contrôles sur les données du projet

- `PASS` : le contrôle est réussi ;
- `WARN` : point à surveiller, les données sont gardées ;
- `INFO` : information sur un traitement ;
- `FAIL` : le contrôle a échoué.

In [2]:
import sqlite3
import pandas as pd
from IPython.display import display

# Lecture seule : le Data Warehouse n'est pas modifié
base = sqlite3.connect((ROOT / "03_data" / "warehouse" / "smartport_complet.db").as_uri() + "?mode=ro", uri=True)
checks = pd.read_sql("SELECT check_id, layer, description, observed, expected, status FROM etl_quality_checks ORDER BY layer, check_id", base)
print(len(checks), "contrôles")
display(checks.status.value_counts())

32 contrôles


status
PASS    30
WARN     1
INFO     1
Name: count, dtype: int64

In [3]:
# Nombre de contrôles par couche et par statut
pd.crosstab(checks.layer, checks.status)

status,INFO,PASS,WARN
layer,,,
CORE,0,9,1
MART,0,1,0
RAW,0,18,0
STAGING,1,2,0


In [4]:
# Détail de tous les contrôles
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 110):
    display(checks)

,check_id,layer,description,observed,expected,status
0,Q02,CORE,Unicite de event_id dans fact_container_event,0.0,0.0,PASS
1,Q03,CORE,Unicite de move_id dans fact_yard_move,0.0,0.0,PASS
2,Q05,CORE,Integrite referentielle : cles de fact_container_event absentes des dimensions,0.0,0.0,PASS
3,Q06,CORE,Integrite referentielle : cles de fact_yard_move absentes des dimensions,0.0,0.0,PASS
4,Q08,CORE,Plage du poids : valeurs hors 0..40000 kg dans fact_container_event,0.0,0.0,PASS
5,Q09,CORE,Reconciliation evenements : RAW - (CORE + rejets),0.0,0.0,PASS
6,Q10,CORE,Reconciliation mouvements : RAW - (CORE + rejets),0.0,0.0,PASS
7,Q11,CORE,Conservation de la mesure : poids total STAGING - poids total CORE (kg),0.0,0.0,PASS
8,Q13,CORE,"Coherence metier : mouvements de cour avec from_area = to_area (signales, conserves)",712.0,0.0,WARN
9,Q14,CORE,Integrite : mouvements portant sur un conteneur absent des evenements (regle FK_CONTAINER),0.0,0.0,PASS


**Interprétation.** Sur 32 contrôles, 30 sont `PASS`, 1 est `WARN` et 1 est `INFO` ; aucun n'est `FAIL`.
- `WARN` (Q13) : 712 mouvements sur 4 500 (15,8 %) ont la même zone de départ et d'arrivée. Ils sont signalés et conservés, car les données ne permettent pas de dire si ce sont des erreurs.
- `INFO` (Q15) : 1 481 événements de la sous-zone Y08B sont rattachés à la zone Y08 dans la colonne de zone standardisée.

In [5]:
# Réconciliation : lignes RAW = lignes chargées + lignes rejetées
recon = pd.read_sql("""
SELECT 'container_events' AS source, (SELECT COUNT(*) FROM raw_container_events) AS raw,
       (SELECT COUNT(*) FROM fact_container_event) AS chargees,
       (SELECT COUNT(DISTINCT raw_rowid) FROM etl_rejects WHERE source_table='container_events') AS rejetees
UNION ALL
SELECT 'yard_moves', (SELECT COUNT(*) FROM raw_yard_moves), (SELECT COUNT(*) FROM fact_yard_move),
       (SELECT COUNT(DISTINCT raw_rowid) FROM etl_rejects WHERE source_table='yard_moves')""", base)
recon["ecart"] = recon.raw - recon.chargees - recon.rejetees
base.close()
recon

,source,raw,chargees,rejetees,ecart
0,container_events,9000,9000,0,0
1,yard_moves,4500,4500,0,0


**Interprétation.** L'écart est nul : les 9 000 événements et les 4 500 mouvements sont tous chargés, avec 0 rejet. Aucune règle de rejet ne se déclenche sur ces données. Pour montrer que les règles fonctionnent, `test_rejets.ipynb` ajoute des erreurs exprès.